# Data Cleaning

This notebook loads the extracted CSVs from `data/02_extraction/`, applies cleaning transformations via `COWCleaner`, and writes clean versions to `data/03_cleaning/`. Cleaning decisions are documented in `src/processing/cleaner.py`; this notebook demonstrates each step individually so intermediate state can be inspected.

## Implementation

Cleaning is implemented by `COWCleaner` in `src/processing/cleaner.py`. Each method is a discrete step that can be called individually for inspection.

| Method | Operation |
|---|---|
| `drop_event_columns()` | Remove spatially redundant and analytically irrelevant event columns |
| `deduplicate_events()` | Keep one row per warning at terminal status (`CAN > COR > EXP > EXT > CON > NEW`) |
| `parse_event_timestamps()` | Parse `issue`/`expire` to UTC datetimes; derive `duration_min` |
| `cap_lead0()` | Cap `lead0` at 99th percentile per phenomena; flag with `lead0_capped` |
| `derive_product_id()` | Add `product_id` join key (`{year}{wfo}{eventid}{phenomena}W1`) |
| `drop_stormreport_columns()` | Remove `type`, `magnitude`, `link` |
| `parse_stormreport_timestamps()` | Parse `valid` to UTC datetime |
| `normalize_source()` | Canonicalize source values; collapse automated stations; impute junk |
| `cap_leadtime()` | Cap `leadtime` at 99th percentile per lsrtype; flag with `leadtime_capped` |
| `impute_null_cities()` | Reverse-geocode null `city` values via Nominatim |
| `impute_null_counties()` | Reverse-geocode null `county` values via Nominatim |

In [1]:
import logging
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../src")

from cleaning import COWCleaner

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)

# --- Configuration ---

# Input: flattened CSVs written by 02_extraction.ipynb
EXTRACTED_DIR = Path("../data/02_extraction")

# Output: cleaned, analysis-ready CSVs for downstream notebooks
PROCESSED_DIR = Path("../data/03_cleaning")

events       = pd.read_csv(EXTRACTED_DIR / "events.csv")
stormreports = pd.read_csv(EXTRACTED_DIR / "stormreports.csv")

print(f"events:       {len(events):,} rows × {len(events.columns)} columns")
print(f"stormreports: {len(stormreports):,} rows × {len(stormreports.columns)} columns")

cleaner = COWCleaner(processed_dir=PROCESSED_DIR)
print(cleaner)

events:       161,481 rows × 32 columns
stormreports: 236,800 rows × 19 columns
COWCleaner(processed=../data/03_cleaning)


## Events

### 1. Drop uninformative columns

Several columns add no analytical value and are dropped before any other work:

- `significance`: constant `'W'` (Warning) across all rows — no variance
- `windtag`: 75% null; wind speed tags are not used in any planned analysis
- `svr_tornado_possible`, `tor_in_svrtorpossible`: SVR-specific flags, not relevant to FF or the main POD/FAR metrics
- `sharedborder`, `carea`, `perimeter`, `parea`, `areaverify`, `lat0`, `lon0`: spatial geometry fields; analysis is statistical, not spatial
- `ar_ugc`, `ar_ugcname`: county/zone codes, not needed at this level of analysis
- `visual_imgurl`, `product_text`, `product_href`, `link`: URL/text blobs with no analytical use
- `stormreports`, `stormreports_all`: embedded JSON lists of storm report IDs; the `stormreports` table is the authoritative source for this relationship
- `fcster`: investigated as a staffing proxy but unusable — format varies within WFOs (badge numbers, last names, initials coexisting), making unique-count comparisons unreliable even within a single office
- `statuses`: identical to `status` after deduplication — a pre-dedup API artifact with no additional information
- `hailtag`: 100% null for FF (structurally inapplicable to flash flood warnings); not used in any planned analysis

In [2]:
events = cleaner.drop_event_columns(events)
print(f"After drop: {len(events.columns)} columns remaining")
print(events.columns.tolist())

13:12:00  INFO     Dropped 22 event columns; 10 remaining


After drop: 10 columns remaining
['wfo', 'year', 'phenomena', 'eventid', 'issue', 'expire', 'product_id', 'status', 'verify', 'lead0']


### 2. Deduplicate events

The COW API returns one row per warning issuance status (`NEW`, `CON`, `EXT`, `EXP`, `CAN`, `COR`). For this analysis each warning is one event, so we keep only the terminal status row per `(wfo, year, phenomena, eventid)`. The terminal status is determined by priority: `CAN` > `COR` > `EXP` > `EXT` > `CON` > `NEW`. This preserves the final verified/cancelled state of each warning.

In [3]:
events = cleaner.deduplicate_events(events)
print(f"After deduplication: {len(events):,} rows")
print("\nStatus distribution after deduplication:")
print(events["status"].value_counts().to_string())

13:12:00  INFO     Deduplicated events: 161,481 → 161,404 rows (removed 77)


After deduplication: 161,404 rows

Status distribution after deduplication:
status
EXP    60731
CON    48039
CAN    34049
NEW    17475
EXT     1078
COR       32


### 3. Parse timestamps

`issue` and `expire` are ISO 8601 strings. Parse them to UTC-aware datetimes and derive `duration_min` (warning duration in minutes) as a convenience column.

In [4]:
events = cleaner.parse_event_timestamps(events)
print("issue range:",   events["issue"].min(),  "→", events["issue"].max())
print("expire range:",  events["expire"].min(), "→", events["expire"].max())
print("duration_min — min:", events["duration_min"].min(), "  max:", events["duration_min"].max())
events[["issue", "expire", "duration_min"]].head(3)

13:12:01  INFO     Parsed issue/expire timestamps; derived duration_min


issue range: 2020-01-02 20:41:00+00:00 → 2025-12-29 06:24:00+00:00
expire range: 2020-01-02 21:30:00+00:00 → 2025-12-29 07:00:00+00:00
duration_min — min: 0.0   max: 9224.0


,issue,expire,duration_min
0,2025-07-31 17:52:00+00:00,2025-07-31 20:20:00+00:00,148.0
1,2021-04-23 18:17:00+00:00,2021-04-23 18:40:00+00:00,23.0
2,2021-04-23 18:43:00+00:00,2021-04-23 19:08:00+00:00,25.0


### 4. Inspect `lead0`

`lead0` is the API-supplied lead time in minutes for the first confirming storm report for each verified warning event. It is already numeric in the extracted CSV — no parsing needed. Non-null for all verified events; null for unverified events.

In [5]:
verified = events[events["verify"] == True]
print(f"lead0 non-null: {verified['lead0'].notna().sum():,} of {len(verified):,} verified events")
print()
print(verified["lead0"].describe(percentiles=[.25, .5, .75, .90, .95, .99]).to_string())

lead0 non-null: 74,679 of 74,679 verified events

count    74679.000000
mean        21.036021
std         33.277759
min          0.000000
25%          5.000000
50%         13.000000
75%         25.000000
90%         42.000000
95%         64.000000
99%        166.000000
max       1241.000000


### 5. Cap extreme `lead0` values

FF has a max of ~7,911 minutes (~5.5 days), almost certainly from the API retroactively matching a warning to a storm report days later due to spatial overlap. TO and SV are naturally bounded. Cap per phenomena at the 99th percentile; `lead0_capped` flags affected rows.

In [6]:
events = cleaner.cap_lead0(events)
print(f"Total capped: {events['lead0_capped'].sum():,}")
print("\nPost-cap distribution by phenomena (verified only):")
verified_mask = events["verify"] == True
print(
    events[verified_mask]
    .groupby("phenomena")["lead0"]
    .describe(percentiles=[.25, .5, .75, .95])
    .to_string()
)

13:12:01  INFO     lead0 cap: FF cap=317 min, 108 rows capped
13:12:01  INFO     lead0 cap: SV cap=53 min, 523 rows capped
13:12:01  INFO     lead0 cap: TO cap=41 min, 41 rows capped


Total capped: 672

Post-cap distribution by phenomena (verified only):
             count       mean        std  min   25%   50%   75%    95%     max
phenomena                                                                     
FF         10730.0  61.230416  61.391512  0.0  19.0  42.0  83.0  186.0  317.42
SV         59734.0  14.277798  12.453125  0.0   4.0  11.0  21.0   40.0   53.00
TO          4215.0  10.373191   9.809480  0.0   2.0   7.0  16.0   31.0   41.00


### 6. Derive join key (`product_id`)

The storm reports table links back to events via a VTEC product ID (e.g. `2020ABQ41SVW1`) stored in the `events` column. The format is `{year}{wfo}{eventid}{phenomena}W1`. Adding this as an explicit column on the events table makes the join unambiguous downstream.

In [7]:
events = cleaner.derive_product_id(events)

# Verify join key coverage
sr_event_ids = (
    stormreports[stormreports["warned"] == True]["events"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)
match_rate = sr_event_ids.isin(events["product_id"]).mean()
print(f"SR → Event match rate via product_id: {match_rate:.2%}")
print("(Remaining misses have significance digit > 1 — co-issued warnings)")
events[["wfo", "year", "phenomena", "eventid", "product_id"]].head(3)

13:12:01  INFO     Derived product_id for 161,404 events


SR → Event match rate via product_id: 99.97%
(Remaining misses have significance digit > 1 — co-issued warnings)


,wfo,year,phenomena,eventid,product_id
0,OKX,2025,FF,15,2025OKX15FFW1
1,EWX,2021,SV,50,2021EWX50SVW1
2,EWX,2021,SV,51,2021EWX51SVW1


### 7. Save cleaned events

In [8]:
cleaner.save(events, "events.csv")
print(f"Saved {len(events):,} rows × {len(events.columns)} columns → data/03_cleaning/events.csv")
print(events.columns.tolist())

13:12:02  INFO     Saved 161,404 rows to ../data/03_cleaning/events.csv


Saved 161,404 rows × 12 columns → data/03_cleaning/events.csv
['wfo', 'year', 'phenomena', 'eventid', 'issue', 'expire', 'product_id', 'status', 'verify', 'lead0', 'duration_min', 'lead0_capped']


## Storm Reports

### 1. Repair malformed rows

A fixed-width field overflow bug in the IEM API corrupts adjacent columns when a long `remark` is serialized, affecting 26 rows across 10 WFOs. Four distinct patterns are repaired in order:

1. **lat0 truncation with clean duplicate** (18 rows) — the corrupt row is dropped; the clean copy is retained
2. **lat0 truncation without clean duplicate** (2 rows — MFL, SGF) — the missing leading digit is inferred from the plausible latitude range and prepended
3. **lon0 sign loss** (5 rows — ILX) — positive CONUS longitude repaired by negating; GUM (Guam/Saipan) with legitimately positive lon0 is excluded
4. **Junk state/county with valid coordinates** (1 row — PIH) — placeholder values nulled so the subsequent Nominatim imputation step resolves them

In [ ]:
print(f"Rows before: {len(stormreports):,}")
stormreports = cleaner.repair_malformed_rows(stormreports)
print(f"Rows after:  {len(stormreports):,}")

### 2. Drop uninformative columns

- `type`: single-character raw code that duplicates `lsrtype`; the latter is more readable
- `magnitude`: 67% null; magnitude values are not used in any planned analysis
- `link`: URL blob with no analytical use

`lon0` and `lat0` are retained for potential spatial EDA. `events` is retained as the trace link from each storm report back to its matched warning(s) — it is a comma-separated list of VTEC product IDs (e.g. `2020ABQ41SVW1`) and is null only for unwarned reports.

In [9]:
stormreports = cleaner.drop_stormreport_columns(stormreports)
print(f"After drop: {len(stormreports.columns)} columns remaining")
print(stormreports.columns.tolist())

13:12:02  INFO     Dropped 3 stormreport columns; 16 remaining


After drop: 16 columns remaining
['wfo', 'year', 'valid', 'city', 'county', 'state', 'source', 'remark', 'typetext', 'lon0', 'lat0', 'events', 'tdq', 'warned', 'leadtime', 'lsrtype']


### 3. Parse timestamp

`valid` is an ISO 8601 string. Parse to UTC-aware datetime.

In [10]:
stormreports = cleaner.parse_stormreport_timestamps(stormreports)
print("valid range:", stormreports["valid"].min(), "→", stormreports["valid"].max())

13:12:02  INFO     Parsed stormreports.valid timestamps


valid range: 2020-01-02 04:45:00+00:00 → 2025-12-29 12:42:00+00:00


### 4. Normalize `source`

Four issues addressed in order:

1. **Case and whitespace** — strip and uppercase collapses 132 → 91 unique values
2. **Truncation artifacts** — values where the leading character(s) were clipped (e.g. `MERGENCY MNGR`, `UBLIC`, `ROADCAST MEDIA`) mapped back to their canonical form
3. **Abbreviation variants** — `COUNTY EMERGY MG` / `COUNTY EMERGY MGMT` → `COUNTY OFFICIAL`; `NEWSPAPER` → `PRINT MEDIA`; `SOCIAL MEDIA` retained as distinct from `BROADCAST MEDIA` and `PRINT MEDIA`; automated station codes (`WEATHERFLOW`, `WXFLOW`, `MESOWEST`, etc.) → `AUTOMATED STATION`
4. **Null and junk values** (`X`, `NONE`, `UNKNOWN`, `FIRE`) — imputed as `"not_provided"`

In [11]:
stormreports = cleaner.normalize_source(stormreports)
print(f"Null source: {stormreports['source'].isnull().sum()}")
print()
print(stormreports["source"].value_counts().to_string())

13:12:02  INFO     Normalized source: 132 → 39 unique values


Null source: 0

source
PUBLIC               64309
EMERGENCY MNGR       33128
911 CALL CENTER      29611
TRAINED SPOTTER      22932
LAW ENFORCEMENT      14704
MESONET              12697
BROADCAST MEDIA       9925
NWS STORM SURVEY      6550
DEPT OF HIGHWAYS      6457
FIRE DEPT/RESCUE      5795
ASOS                  4711
AMATEUR RADIO         4707
NWS EMPLOYEE          4387
UTILITY COMPANY       3880
STORM CHASER          2562
SOCIAL MEDIA          2092
COUNTY OFFICIAL       1963
AWOS                  1902
CO-OP OBSERVER        1221
OTHER FEDERAL          841
COCORAHS               703
OFFICIAL NWS OBS       583
PARK/FOREST SRVC       546
PRINT MEDIA            361
AUTOMATED STATION       66
POST OFFICE             50
not_provided            42
TOWN OFFICE             18
AIRPLANE PILOT          17
NWS OFFICE              15
TRIBAL OFFICIAL         13
COAST GUARD              3
INSURANCE CO             3
NM AIR QUALITY           1
NYS DEC                  1
GOLF COURSE              1
USGS 

### 5. Inspect `leadtime`

`leadtime` is the API-supplied lead time in minutes between warning issuance and each storm report. It is already numeric in the extracted CSV — no parsing needed. Non-null for all warned reports; null for unwarned reports.

In [12]:
warned_sr = stormreports[stormreports["warned"] == True]
print(f"leadtime non-null: {warned_sr['leadtime'].notna().sum():,} of {len(warned_sr):,} warned rows")
print()
print(warned_sr["leadtime"].describe(percentiles=[.25, .5, .75, .90, .95, .99]).to_string())

leadtime non-null: 181,826 of 181,826 warned rows

count    181826.000000
mean         35.351451
std          73.535475
min           0.000000
25%          12.000000
50%          23.000000
75%          38.000000
90%          58.000000
95%         109.000000
99%         277.000000
max        7996.000000


### 6. Cap extreme `leadtime` values

`leadtime` is API-supplied, computed server-side as the difference in minutes between warning issuance and storm report time. FF warnings can spatially overlap storm reports from days later, producing physically implausible values (max ~7,996 min). TO and SV are naturally bounded. Cap per lsrtype at the 99th percentile; `leadtime_capped` flags affected rows.

In [13]:
stormreports = cleaner.cap_leadtime(stormreports)
print(f"Total capped: {stormreports['leadtime_capped'].sum():,}")
print("\nPost-cap distribution by lsrtype (warned only):")
warned_mask = stormreports["warned"] == True
print(
    stormreports[warned_mask]
    .groupby("lsrtype")["leadtime"]
    .describe(percentiles=[.25, .5, .75, .95])
    .to_string()
)

13:12:02  INFO     leadtime cap: FF cap=520 min, 266 rows capped
13:12:02  INFO     leadtime cap: SV cap=62 min, 1,295 rows capped
13:12:02  INFO     leadtime cap: TO cap=49 min, 66 rows capped


Total capped: 1,627

Post-cap distribution by lsrtype (warned only):
            count        mean        std  min   25%   50%    75%    95%     max
lsrtype                                                                        
FF        26535.0  104.431489  99.582147  0.0  35.0  73.0  141.0  311.0  519.66
SV       148344.0   23.088194  14.524638  0.0  11.0  21.0   33.0   50.0   62.00
TO         6947.0   15.501799  11.868901  0.0   6.0  13.0   23.0   38.0   49.00


### 7. Flag TDQ reports

`tdq` ("Too Difficult to Qualify") marks reports that NWS could not verify as a true hazardous event. These are retained in the dataset but flagged — downstream analyses that compute POD should consider excluding them to avoid inflating verified event counts.

In [14]:
tdq_count = stormreports["tdq"].sum()
print(f"TDQ reports: {tdq_count:,} of {len(stormreports):,} ({tdq_count/len(stormreports):.1%})")
print()
print("TDQ by lsrtype:")
print(stormreports.groupby("lsrtype")["tdq"].sum().to_string())

TDQ reports: 4,408 of 236,800 (1.9%)

TDQ by lsrtype:
lsrtype
FF      58
SV    4350
TO       0


### 8. Impute null `city` values

A small number of rows have no city. For each, reverse-geocode `lon0`/`lat0` via the Nominatim API and use the most specific named place available (city → town → village → hamlet), falling back to `"not_provided"` if the location is too rural to resolve.

In [15]:
print(f"Null city before: {stormreports['city'].isnull().sum()}")
stormreports = cleaner.impute_null_cities(stormreports)
print(f"Null city after:  {stormreports['city'].isnull().sum()}")

13:12:02  INFO     Imputing 2 null city values via Nominatim ...


Null city before: 2


13:12:02  INFO       city imputed: idx=71449 (43.1, -96.19) → 'Sioux Center'
13:12:03  INFO       city imputed: idx=71472 (44.12, -95.74) → 'not_provided'


Null city after:  0


### 9. Impute null `county` values

18 rows (all AJK 2020, southeast Alaska) have no county. Alaska uses boroughs rather than counties, and some areas are part of the Unorganized Borough — Nominatim may return `county`, `city` (for unified city-boroughs like Juneau), or nothing. Use the same reverse-geocode approach, falling back through `county` → `city` → `state_district` → `"not_provided"`.

In [16]:
print(f"Null county before: {stormreports['county'].isnull().sum()}")
stormreports = cleaner.impute_null_counties(stormreports)
print(f"Null county after:  {stormreports['county'].isnull().sum()}")

13:12:05  INFO     Imputing 18 null county values via Nominatim ...


Null county before: 18


13:12:05  INFO       county imputed: idx=3743 (55.36, -131.69) → 'Ketchikan Gateway Borough'
13:12:06  INFO       county imputed: idx=3747 (56.39, -132.24) → 'Wrangell'
13:12:08  INFO       county imputed: idx=3748 (55.62, -132.93) → 'Unorganized Borough'
13:12:10  INFO       county imputed: idx=3749 (55.55, -133.08) → 'Unorganized Borough'
13:12:12  INFO       county imputed: idx=3752 (55.48, -133.14) → 'Unorganized Borough'
13:12:13  INFO       county imputed: idx=3753 (58.3, -134.42) → 'Juneau'
13:12:14  INFO       county imputed: idx=3754 (59.25, -135.44) → 'Haines Borough'
13:12:16  INFO       county imputed: idx=3755 (59.38, -135.84) → 'Haines Borough'
13:12:17  INFO       county imputed: idx=3756 (59.42, -136.02) → 'Haines Borough'
13:12:19  INFO       county imputed: idx=3757 (58.32, -134.46) → 'Juneau'
13:12:24  INFO       county imputed: idx=3758 (58.37, -134.56) → 'Juneau'
13:12:26  INFO       county imputed: idx=3759 (59.25, -135.54) → 'Haines Borough'
13:12:27  INFO       

Null county after:  0


### 10. Save cleaned storm reports

In [17]:
cleaner.save(stormreports, "stormreports.csv")
print(f"Saved {len(stormreports):,} rows × {len(stormreports.columns)} columns → data/03_cleaning/stormreports.csv")
print(stormreports.columns.tolist())

13:12:36  INFO     Saved 236,800 rows to ../data/03_cleaning/stormreports.csv


Saved 236,800 rows × 17 columns → data/03_cleaning/stormreports.csv
['wfo', 'year', 'valid', 'city', 'county', 'state', 'source', 'remark', 'typetext', 'lon0', 'lat0', 'events', 'tdq', 'warned', 'leadtime', 'lsrtype', 'leadtime_capped']


## Diagnostics

In [18]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 161404 entries, 0 to 161403
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   wfo           161404 non-null  str                
 1   year          161404 non-null  int64              
 2   phenomena     161404 non-null  str                
 3   eventid       161404 non-null  int64              
 4   issue         161404 non-null  datetime64[us, UTC]
 5   expire        161404 non-null  datetime64[us, UTC]
 6   product_id    161404 non-null  str                
 7   status        161404 non-null  str                
 8   verify        161404 non-null  bool               
 9   lead0         74679 non-null   float64            
 10  duration_min  161404 non-null  float64            
 11  lead0_capped  161404 non-null  bool               
dtypes: bool(2), datetime64[us, UTC](2), float64(2), int64(2), str(4)
memory usage: 12.6 MB


In [19]:
stormreports.info()

<class 'pandas.DataFrame'>
RangeIndex: 236800 entries, 0 to 236799
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   wfo              236800 non-null  str                
 1   year             236800 non-null  int64              
 2   valid            236800 non-null  datetime64[us, UTC]
 3   city             236800 non-null  str                
 4   county           236800 non-null  str                
 5   state            236800 non-null  str                
 6   source           236800 non-null  str                
 7   remark           209678 non-null  str                
 8   typetext         236800 non-null  str                
 9   lon0             236800 non-null  float64            
 10  lat0             236800 non-null  float64            
 11  events           189665 non-null  str                
 12  tdq              236800 non-null  bool               
 13  warned    